Upload document
   ↓
Chunk / split into sentences
   ↓
spaCy → entities + keywords
   ↓
BERT → score sentences
   ↓
Boost scores using:
   - section importance
   - keyword presence
   - entity presence
   ↓
Select top sentences
   ↓
mBART → generate final summary

In [135]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
import spacy
import json
import re
import time
from pathlib import Path

In [136]:
df = pd.read_json("../Data/silver/doc_01_silver_nlp.json")

In [137]:
BERT_MODEL = "google-bert/bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL)
bert_model.eval()

# mBART for final abstractive summary
MBART_MODEL = "facebook/mbart-large-50"
mbart_tokenizer = AutoTokenizer.from_pretrained(
    MBART_MODEL,
    src_lang="en_XX",
    use_fast=False
)
mbart_model = AutoModelForSeq2SeqLM.from_pretrained(MBART_MODEL)

# spaCy
nlp = spacy.load("en_core_web_sm")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21563.15it/s]
BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 519/519 [00:00<00:00, 252066.21it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are pr

In [138]:
RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology"
]

GENERIC_RESEARCH_KEYWORDS = [
    "objective", "objectives",
    "aim", "aims",
    "goal", "goals",
    "purpose",
    "research question", "research questions",
    "method", "methods", "methodology",
    "approach",
    "experiment", "experiments",
    "analysis",
    "result", "results",
    "finding", "findings",
    "discussion",
    "conclusion", "conclusions",
    "recommendation", "recommendations",
    "implication", "implications",
    "evaluation",
    "framework",
    "model",
    "study",
    "project"
]

SECTION_WEIGHTS = {
    "abstract": 1.08,
    "introduction": 1.04,
    "background": 1.00,
    "methods": 0.98,
    "methodology": 0.98,
    "results": 1.06,
    "discussion": 1.05,
    "conclusion": 1.07,
    "conclusions": 1.07,
    "summary": 1.06,
    "default": 1.00
}


In [139]:
def simple_tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def get_bert_embedding(text, max_length=512):
    inputs = bert_tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    return embedding.squeeze(0)

def cosine_similarity(vec1, vec2):
    return torch.nn.functional.cosine_similarity(
        vec1.unsqueeze(0), vec2.unsqueeze(0)
    ).item()

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", str(text)).strip()


In [140]:
def is_good_sentence(sentence):
    s = normalize_whitespace(sentence)

    if len(s) < 50:
        return False
    if len(s.split()) < 8:
        return False
    if re.search(r'\bPage\b|\bP\s*a\s*g\s*e\b', s, flags=re.IGNORECASE):
        return False
    if re.search(r'https?://|www\.', s):
        return False
    if re.search(r'\bet al\.', s, flags=re.IGNORECASE):
        return False
    if re.search(r'\(\s*[A-Z][A-Za-z]+.*\d{4}.*\)', s):
        return False
    if re.fullmatch(r'[\W\d\s]+', s):
        return False

    return True

In [141]:
def detect_sections(sentences):
    current_section = "default"
    sectioned_sentences = []

    for sent in sentences:
        s = sent.strip()
        s_lower = s.lower()

        if len(s.split()) <= 6:
            for section_name in SECTION_WEIGHTS.keys():
                if section_name != "default" and section_name in s_lower:
                    current_section = section_name
                    break

        sectioned_sentences.append({
            "sentence": s,
            "section": current_section
        })

    return sectioned_sentences

In [142]:
def extract_entities_spacy(text):
    doc = nlp(str(text))
    return [(ent.text.strip(), ent.label_) for ent in doc.ents]

def extract_key_entities(text, max_chars=5000):
    front_text = str(text)[:max_chars]
    doc = nlp(front_text)

    allowed_labels = {"PERSON", "ORG", "GPE", "PRODUCT", "NORP"}
    entities = []

    for ent in doc.ents:
        ent_text = normalize_whitespace(ent.text)

        if ent.label_ not in allowed_labels:
            continue
        if len(ent_text) < 3:
            continue
        if len(ent_text.split()) > 4:
            continue
        if re.search(r'\b(19|20)\d{2}\b', ent_text):
            continue
        if "et al" in ent_text.lower():
            continue
        if re.search(r'[•()\[\]]', ent_text):
            continue

        entities.append(ent_text)

    return sorted(set(entities))

def extract_dynamic_keywords(text, top_n=15):
    doc = nlp(str(text))
    candidates = []

    for chunk in doc.noun_chunks:
        phrase = normalize_whitespace(chunk.text).lower()
        if len(phrase) > 3:
            candidates.append(phrase)

    for ent in doc.ents:
        phrase = normalize_whitespace(ent.text).lower()
        if len(phrase) > 2:
            candidates.append(phrase)

    freq = {}
    for c in candidates:
        freq[c] = freq.get(c, 0) + 1

    ranked = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    return [term for term, _ in ranked[:top_n]]

def build_keyword_list(text):
    dynamic_keywords = extract_dynamic_keywords(text, top_n=15)
    return sorted(set(GENERIC_RESEARCH_KEYWORDS + dynamic_keywords))

def keyword_boost(sentence, keywords):
    s = sentence.lower()
    score = 0.0

    for kw in keywords:
        if kw in s:
            if len(kw.split()) > 1:
                score += 0.20
            else:
                score += 0.10

    return score

def entity_boost(sentence, key_entities):
    s = sentence.lower()
    score = 0.0

    for ent in key_entities:
        if ent.lower() in s:
            if len(ent.split()) > 1:
                score += 0.12
            else:
                score += 0.06

    return score

def section_boost(section_name):
    return SECTION_WEIGHTS.get(section_name, SECTION_WEIGHTS["default"])


In [143]:
def score_sentences_guided(text, sentences):
    doc_embedding = get_bert_embedding(text)
    sectioned = detect_sections(sentences)

    key_entities = extract_key_entities(text)
    keyword_list = build_keyword_list(text)

    scored = []

    for item in sectioned:
        sent = item["sentence"]
        section = item["section"]

        if not sent or not sent.strip():
            continue
        if not is_good_sentence(sent):
            continue

        sent_embedding = get_bert_embedding(sent)
        bert_score = cosine_similarity(doc_embedding, sent_embedding)
        kw_score = keyword_boost(sent, keyword_list)
        ent_score = entity_boost(sent, key_entities)
        sec_weight = section_boost(section)

        final_score = (
            bert_score * 0.65 +
            kw_score * 0.20 +
            ent_score * 0.10 +
            (sec_weight - 1.0) * 0.05
        )

        scored.append({
            "sentence": sent,
            "section": section,
            "bert_score": bert_score,
            "keyword_boost": kw_score,
            "entity_boost": ent_score,
            "section_weight": sec_weight,
            "final_score": final_score
        })

    return sorted(scored, key=lambda x: x["final_score"], reverse=True)

In [144]:
def jaccard_similarity(a, b):
    set_a = set(simple_tokenize(a))
    set_b = set(simple_tokenize(b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def select_top_sentences(scored_sentences, original_sentences, top_n=12, max_similarity=0.65, max_per_section=2):
    selected = []
    section_counts = {}

    for item in scored_sentences:
        sent = item["sentence"]
        section = item["section"]

        if section_counts.get(section, 0) >= max_per_section:
            continue

        if any(jaccard_similarity(sent, prev) > max_similarity for prev in selected):
            continue

        selected.append(sent)
        section_counts[section] = section_counts.get(section, 0) + 1

        if len(selected) >= top_n:
            break

    ordered_selected = [s for s in original_sentences if s in selected]
    return ordered_selected

In [145]:
def summarize_with_mbart(text, max_input_len=1024, max_output_len=280, min_output_len=120):
    inputs = mbart_tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    )

    with torch.no_grad():
        summary_ids = mbart_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_len,
            min_length=min_output_len,
            num_beams=5,
            length_penalty=1.1,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return mbart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [146]:
def is_valid_person_name(name):
    name = normalize_whitespace(name)

    if len(name) < 5:
        return False
    if len(name.split()) < 2:
        return False
    if len(name.split()) > 4:
        return False
    if re.search(r'\b(19|20)\d{2}\b', name):
        return False
    if "et al" in name.lower():
        return False
    if re.search(r'[•()\[\]{}:;\\/]', name):
        return False
    if any(char.isdigit() for char in name):
        return False

    parts = name.split()
    if not all(part[:1].isupper() for part in parts if part):
        return False

    return True

def extract_people(text, max_chars=4000):
    front_text = str(text)[:max_chars]
    doc = nlp(front_text)

    people = []
    for ent in doc.ents:
        if ent.label_ != "PERSON":
            continue

        name = normalize_whitespace(ent.text)

        if not is_valid_person_name(name):
            continue

        people.append(name)

    return sorted(set(people))

def extract_dates(text):
    iso_dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", str(text))
    return sorted(set(iso_dates))

def extract_research_groups(text, research_groups):
    text_lower = str(text).lower()
    matches = []

    for group in research_groups:
        if group.lower() in text_lower:
            matches.append(group)

    return sorted(set(matches))

def is_table_of_contents_line(line):
    line_clean = normalize_whitespace(line).lower()

    if "table of contents" in line_clean:
        return True
    if re.search(r'\.{4,}\s*\d+$', line):
        return True
    if re.match(r'^\d+(\.\d+)*\s+.+\s+\d+$', line_clean):
        return True

    return False


def extract_title(text):
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    lines = lines[:30]

    filtered = []
    for line in lines:
        if is_table_of_contents_line(line):
            continue
        if len(line.split()) < 3:
            continue
        if len(line) > 180:
            continue
        if is_valid_person_name(line):
            continue
        filtered.append(line)

    if not filtered:
        return lines[0] if lines else ""

    # Prefer an early meaningful line, not the longest one
    return filtered[0]

def extract_description(summary, fallback_text):
    if summary and str(summary).strip():
        return str(summary).strip()
    return str(fallback_text)[:500].strip()

def extract_contact_person(text):
    patterns = [
        r'contact person[:\s]+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)',
        r'author[:\s]+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)',
        r'supervisor[:\s]+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)'
    ]

    front_text = str(text)[:4000]

    for pattern in patterns:
        match = re.search(pattern, front_text, flags=re.IGNORECASE)
        if match:
            candidate = normalize_whitespace(match.group(1))
            if is_valid_person_name(candidate):
                return candidate

    return ""

def extract_project_metadata(text, summary=None, document_id=None):
    people = extract_people(text)
    dates = extract_dates(text)
    research_groups = extract_research_groups(text, RESEARCH_GROUPS)

    metadata = {
        "id": document_id or "",
        "title": extract_title(text),
        "description": extract_description(summary, text),
        "contact_person": extract_contact_person(text),
        "start_date": dates[0] if len(dates) > 0 else "",
        "end_date": dates[1] if len(dates) > 1 else "",
        "research_groups": research_groups,
        "researchers": people
    }

    return metadata


In [147]:
def evaluate_summary(original_text, summary_text, source_entities=None):
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)

    compression_ratio = summary_len / original_len if original_len > 0 else 0

    preserved_entities = []
    missed_entities = []

    source_entities = source_entities or []
    summary_lower = str(summary_text).lower()

    for ent in source_entities:
        if isinstance(ent, dict):
            ent_text = ent.get("text", "")
        elif isinstance(ent, (list, tuple)) and len(ent) >= 1:
            ent_text = ent[0]
        else:
            ent_text = str(ent)

        ent_text_clean = normalize_whitespace(ent_text)

        if ent_text_clean and ent_text_clean.lower() in summary_lower:
            preserved_entities.append(ent_text_clean)
        else:
            missed_entities.append(ent_text_clean)

    total_entities = len(source_entities)
    entity_preservation_score = (
        len(preserved_entities) / total_entities if total_entities > 0 else None
    )

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }

def sentence_coverage_score(source_sentences, summary_text, threshold=0.2):
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue

        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0

In [148]:
row = df.loc[0]
document_id = row["document_id"]
text = row["cleaned_text"]
sentences = [s for s in row["sentences"] if is_good_sentence(s)]

start = time.time()

scored_sentences = score_sentences_guided(text, sentences)
selected_sentences = select_top_sentences(scored_sentences, sentences, top_n=20)

guided_extract = " ".join(selected_sentences)
final_summary = summarize_with_mbart(
    guided_extract,
    max_input_len=1024,
    max_output_len=480,
    min_output_len=160
)

runtime_seconds = time.time() - start

metadata = extract_project_metadata(
    text=text,
    summary=final_summary,
    document_id=document_id
)

eval_result = evaluate_summary(
    original_text=text,
    summary_text=final_summary,
    source_entities=row["entities"]
)
eval_result["runtime_seconds"] = runtime_seconds
eval_result["sentence_coverage_score"] = sentence_coverage_score(sentences, final_summary)

combined_output = {
    "document_id": document_id,
    "model": "guided_BERT_mBART",
    "guided_extract": guided_extract,
    "selected_sentences": selected_sentences,
    "sentence_scores": scored_sentences,
    "summary": final_summary,
    "metadata": metadata,
    "evaluation": {
        "runtime_seconds": runtime_seconds,
        "original_token_count": eval_result["original_token_count"],
        "summary_token_count": eval_result["summary_token_count"],
        "compression_ratio": eval_result["compression_ratio"],
        "entity_preservation_score": eval_result["entity_preservation_score"],
        "sentence_coverage_score": eval_result["sentence_coverage_score"]
    }
}

In [149]:
output_dir = Path("../Data/gold/guided_BERT_mBART")
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "guided_BERT_mBART_output.json", "w", encoding="utf-8") as f:
    json.dump(combined_output, f, indent=4, ensure_ascii=False)

print("Saved guided BERT + mBART output.")
print(final_summary)

Saved guided BERT + mBART output.
Data Analysis / Results .............................................................................................. 9 3.1. Explanation and consideration ................................................................. Data quality report of the data ............................................................. 17 4. 4. Conclusion ............................................................. 17 5. Conclusion of the study ....................................................................................... 19 6. Conclusions .......................................................................... 19 7. 8. 9. 10. 11. 11. 12. 12. 12. 13. 13. 14. 14............................................................................Data Analysis/Results/Results/Contents/References/Reviews/Downloads/Files/DATA/Specifications/Comparison/Resources/Data/Relations/Transactions/Descriptions/Documents/Sitemaps/
